In [ ]:
import sys
import os
# Add the src directory to the Python path
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.append(project_root)
import json
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from src.utils.plotting import _save_figure
from src.utils import (
    load_fitted_results,
    plot_compare_n_models_latex,
    plot_shortlong_vs_combined_latex,
    plot_compare_two_models_latex,
    plot_compare_three_models_latex,
    get_best_model,
)
from collections import Counter
import glob
import math

In [ ]:
report_figures_path= os.path.join(project_root, 'BIC_commit', 'figures')
if not os.path.exists(report_figures_path):
    os.makedirs(report_figures_path)
report_tables_path= os.path.join(project_root, 'BIC_commit', 'tables')
if not os.path.exists(report_tables_path):
    os.makedirs(report_tables_path)

# Single global switch (auto-detected, override in src/utils/plotting.py if needed)
# instead of a local hardcoded value -- this and its commit sibling used to disagree
# (True vs False) since each had its own copy.
from src.utils.plotting import IS_LATEX as is_latex

In [ ]:
figure_height = 6
# _set_plot_style now lives in src.utils.plotting (IS_LATEX=False there); the
# plotting functions imported above already use it internally with is_latex passed through.

In [ ]:
config_dir = os.path.join(project_root, "data/simulation_configs")

results_df_dict, results_df_recovered_dict = load_fitted_results(config_dir, commit=True)

In [ ]:
results_df_dict.keys()

In [ ]:
import os
import pandas as pd

# Import your new loader
from src.config.loader import load_config
config_files = glob.glob(f"{config_dir}/*.py")
def generate_simulation_params_table(config_files, report_tables_path):
    """
    Generates a LaTeX table of simulation parameters from configuration files,
    sorted by model type (S, L, C).
    Model naming convention: (S/L/C) + NBXTGRPHCLUK
    Filename format: simulation_params_CBEXT-RPHCLUK.py
    """
    table_data = []

    for file_path in config_files:
        # 1. Load the config object using your new loader
        try:
            config = load_config(override_path=file_path)
        except AttributeError:
            # Skip files that aren't valid override configs
            continue

        # Extract model name and horizon condition directly from filename
        filename = os.path.basename(file_path)[:-3]          # remove .py
        key = filename.replace("simulation_params_", "")     # e.g. "CNEXT-RPHCLUK"
        M = key[0]                                           # S, L, or C
        latex_model_name = f"\\texttt{{{key}}}"

        # 2. Extract values directly from the config dataclass instance
        fixed_params = {
            "TAU":                  config.TAU,
            "GAMMA":                config.GAMMA,
            "EXAGGERATION_FACTOR": config.EXAGGERATION_FACTOR,
            "XI":                   config.XI,
            "BELIEF_BIAS":          config.BELIEF_BIAS,
            "IS_HAZARDOUS":         config.IS_HAZARDOUS,
            "SUBJECTIVE_COST":      config.SUBJECTIVE_COST,
            "PATIENCE":             config.PATIENCE,
            "URGENCY_COEFFICIENT":  config.URGENCY_COEFFICIENT,
            "URGENCY_SLOPE":        config.URGENCY_SLOPE,
            "C_MAX":                config.C_MAX,
            "HAZARD_LAPSE":         config.HAZARD_LAPSE,
        }
        param_ranges = config.PARAM_RANGES
        param_order  = config.PARAM_ORDER

        param_values = {}
        for p in fixed_params:
            pname = p.lower()
            if pname in param_ranges:
                param_values[pname] = f"$({param_ranges[pname][0]},{param_ranges[pname][1]})$"
            else:
                val = fixed_params[p]
                param_values[pname] = f"$\\text{{{val}}}$" if isinstance(val, str) else f"${val}$"

        row = {
            "Model":              latex_model_name,
            r"$N_\text{params}$": len(param_order),
            "_sort_key":          M,
        }
        row.update(param_values)
        table_data.append(row)

    # Create the DataFrame
    df = pd.DataFrame(table_data).fillna("-")

    # Sort by model type (S, L, C)
    sort_order = ['S', 'L', 'C']
    df['_sort_key'] = pd.Categorical(df['_sort_key'], categories=sort_order, ordered=True)
    df.sort_values('_sort_key', inplace=True)
    df.drop(columns=['_sort_key'], inplace=True)

    # Rename columns for LaTeX following NEXTGRPHCLUK order. Symbols match
    # PARAM_LABEL_MAP in src/utils/plotting.py (the same notation used for
    # every plot axis label in the paper), so this table's column headers
    # are consistent with the rest of the manuscript -- in particular
    # patience/c_max/urgency_coefficient use $t_p$/$\phi_{\max}$/$\phi_{\min}$
    # here, not the shorter $p$/$c_{\max}$/$u$ used informally elsewhere in
    # this notebook.
    rename_map = {
        "belief_bias":          r"$B$",
        "exaggeration_factor": r"$E$",
        "xi":                   r"$\xi$",
        "tau":                  r"$\tau$",
        "gamma":                r"$\gamma$",
        "subjective_cost":      r"$R_{\mathrm{risk}}$",
        "patience":             r"$t_p$",
        "is_hazardous":         r"$H$",
        "c_max":                r"$\phi_{\max}$",
        "hazard_lapse":         r"$L$",
        "urgency_coefficient":  r"$\phi_{\min}$",
        "urgency_slope":        r"$k$",
    }
    df.rename(columns=rename_map, inplace=True)

    # Define column order following NBXTGRPHCLUK
    ordered_columns = [
        "Model", r"$N_\text{params}$",
        r"$B$", r"$E$", r"$\xi$", r"$\tau$", r"$\gamma$",
        r"$R_{\mathrm{risk}}$", r"$t_p$", r"$H$", r"$\phi_{\max}$", r"$L$",
        r"$\phi_{\min}$", r"$k$",
    ]
    df = df[[c for c in ordered_columns if c in df.columns]]

    # LaTeX export
    latex_table = df.to_latex(
        index=False, escape=False, column_format='l' + 'c' * (len(df.columns) - 1)
    )
    latex_table = (
        "\\resizebox{0.97\\textwidth}{!}{%\n" +
        latex_table + "}\n"
    )

    output_file = os.path.join(report_tables_path, "simulation_params_table.tex")
    with open(output_file, "w") as f:
        f.write(latex_table)

    print(f"LaTeX table exported as {output_file}, sorted by model type (S, L, C).")

In [ ]:

generate_simulation_params_table(config_files, report_tables_path)

In [ ]:
# ==========================================
# EXECUTION SCRIPT
# ==========================================

# 1. Dynamically group all available models by their starting letter
model_list = list(results_df_dict.keys())
short_models = [m for m in model_list if m.startswith('S')]
long_models = [m for m in model_list if m.startswith('L')]
combined_models = [m for m in model_list if m.startswith('C')]

print(f"Found {len(short_models)} Short models, {len(long_models)} Long models, and {len(combined_models)} Combined models.")

# 2. Compare ALL short models
results_short, best_model_short = plot_compare_n_models_latex(
    model_names=short_models,
    results_df_dict=results_df_dict,
    is_latex=is_latex,
    path=report_figures_path,top_n=10,
    export_table_path=os.path.join(report_tables_path, "short_models_metrics.tex")
)
print(results_short["metrics_table"])

# 3. Compare ALL long models
results_long, best_model_long = plot_compare_n_models_latex(
    model_names=long_models,
    results_df_dict=results_df_dict,
    is_latex=is_latex,
    path=report_figures_path,
    export_table_path=os.path.join(report_tables_path, "long_models_metrics.tex"),
    top_n=10,
)
print(results_long["metrics_table"])

# 4. Compare ALL combined models
results_combined, best_model_combined = plot_compare_n_models_latex(
    model_names=combined_models,
    results_df_dict=results_df_dict,
    is_latex=is_latex,
    path=report_figures_path,
    export_table_path=os.path.join(report_tables_path, "combined_models_metrics.tex"),
    break_at=False,top_n=10
)
print(results_combined["metrics_table"])

# 5. Export the best short/long/combined model names so other scripts
#    (e.g. the GLM ensemble driver) can pick them up without re-running this notebook.
best_models = {"short": best_model_short, "long": best_model_long, "combined": best_model_combined}
best_models_path = os.path.join(project_root, "BIC_commit", "best_models.json")
with open(best_models_path, "w") as f:
    json.dump(best_models, f, indent=2)
print(f"Best models exported to {best_models_path}")

# 6. Also export the same winners as LaTeX \newcommand macros (\BestShortModel,
#    \BestLongModel, \BestCombinedModel) so the paper can \input{} this file
#    once and reference the winning model names in prose without hand-typing
#    them -- they stay in sync automatically whenever this notebook reruns.
from src.utils import export_best_models_macros
best_models_macros_path = os.path.join(report_tables_path, "best_models_macros.tex")
export_best_models_macros(best_models, best_models_macros_path)
print(f"Best models LaTeX macros exported to {best_models_macros_path}")

In [ ]:
# 5. Compare the BEST short + BEST long vs BEST combined
# AIC/AICc/BIC here are raw (not delta) values in the tens of thousands, so a
# 0-anchored bar chart hides the combined-vs-separate difference. y_min zooms
# the left panel in on where the bars actually differ; adjust to taste.
results = plot_shortlong_vs_combined_latex(
    results_df_dict,
    key_short=best_model_short,
    key_long=best_model_long,
    key_combined=best_model_combined,
    is_latex=is_latex,
    path=report_figures_path,
    export_table_path=os.path.join(report_tables_path, "comparison_combined_vs_separate.tex"),
    y_min=50000,
)

## GLM (combined) vs POMDP (separate short+long)

GLM is only ever fit jointly across both horizons (no separate-horizon GLM exists), so the comparison here is: the existing combined GLM against the POMDP's best-short + best-long separate fit (the `f"{best_model_short}+{best_model_long}"` row already computed by `plot_shortlong_vs_combined_latex` above), using the identical aggregation method (`BIC = k*log(n_obs) - 2*sum(logL)`, summed across subjects).

In [ ]:
import numpy as np
import pandas as pd
from src.glm import fit_glm_separate_for_human_data
from src.glm.glm import compute_full_per_draw_probabilities
from src.utils.plotting import compute_metrics

human_data_path = os.path.join(project_root, "data/TrHu_NHB_light/data_MEG/behdat_preprocessed.pkl")
glm_human_data = pd.read_pickle(human_data_path)

print("Fitting GLM (combined, jointly across both horizons)...")
betas_glm_combined, _ = fit_glm_separate_for_human_data(glm_human_data, ocir_all=None)

glm_combined_by_id = {
    b["id"]: b for b in betas_glm_combined
    if b is not None and not np.isnan(b["pdecide_beta"]).any() and not np.isnan(b["llf"])
}
print(f"{len(glm_combined_by_id)}/{len(betas_glm_combined)} subjects with a valid GLM fit.")

def build_full_decisions(row_data, games_lengths):
    """
    Binary decision vector aligned to games_lengths, covering ALL raw draws
    (not valid_mask-filtered). y=0 on every draw before the decision,
    y=1 at the decision draw (game_len-1 by construction of games_lengths),
    or all zeros if the subject never decided in that game.
    Length == sum(games_lengths), matching compute_full_per_draw_probabilities.
    """
    decisions = []
    for (_, game_data), game_len in zip(row_data.groupby(["block", "game"]), games_lengths):
        game_dec = np.zeros(game_len, dtype=float)
        decision_index = game_data["choiceTrial"].first_valid_index()
        if decision_index is not None:
            decision_pos = game_data.index.get_loc(decision_index)
            if decision_pos < game_len:
                game_dec[decision_pos] = 1.0
        decisions.append(game_dec)
    return np.concatenate(decisions)

# Recompute LL and n_obs over ALL raw draws, replacing b["llf"] (which is
# only computed on valid_mask-filtered trials inside fit_glm_separate_for_human_data).
for uid, b in glm_combined_by_id.items():
    row_data = glm_human_data.loc[glm_human_data["userID"] == uid, "data"].iloc[0]
    p = compute_full_per_draw_probabilities(
        row_data, b["games_lengths"], b["mu"], b["sigma"], b["pdecide_beta"]
    )
    y = build_full_decisions(row_data, b["games_lengths"])
    p_clipped = np.clip(p, 1e-10, 1 - 1e-10)
    b["ll_raw"]    = float(np.sum(y * np.log(p_clipped) + (1 - y) * np.log(1 - p_clipped)))
    b["n_obs_raw"] = int(sum(b["games_lengths"]))

N_PARAMS_PER_SUBJECT = 7  # 6 regressors (totevminus, deltaev, trial, termination, + 2 interactions) + intercept

# Per-subject-summed convention (matches results["metrics_table"]'s POMDP
# rows below, via compute_metrics_per_subject_summed / plot_shortlong_vs_combined_latex):
# each subject is its own independent GLM fit (own ll_i, own n_obs_i, k=7),
# so the population-level BIC/AIC is the *sum* of each subject's own
# compute_metrics(...) result -- not compute_metrics() computed once from
# pooled ll/n_obs with k_total=7*105. AIC is identical either way (no
# n_obs term), but BIC is not: log(sum(n_i)) != sum(log(n_i)) since n_obs
# varies by subject. This previously used the pooled form, which silently
# put the GLM row on a different aggregation convention than the POMDP rows
# in the same comparison table (bic_table_commit below).
per_subject_glm_metrics = [
    compute_metrics(b["ll_raw"], N_PARAMS_PER_SUBJECT, b["n_obs_raw"])
    for b in glm_combined_by_id.values()
]
metrics_glm_combined = {
    "sum logL": sum(m["sum logL"] for m in per_subject_glm_metrics),
    "k": N_PARAMS_PER_SUBJECT,  # per-subject count, matching the POMDP rows' convention
    # Total observation count across subjects, kept for display/downstream
    # compatibility only -- it plays no role in the per-subject-summed
    # AIC/BIC above (each subject's own n_obs_i already entered its own
    # compute_metrics() call).
    "n_obs": sum(m["n_obs"] for m in per_subject_glm_metrics),
    "AIC": sum(m["AIC"] for m in per_subject_glm_metrics),
    "AICc": sum(m["AICc"] for m in per_subject_glm_metrics),
    "BIC": sum(m["BIC"] for m in per_subject_glm_metrics),
}
print()
print(pd.Series(metrics_glm_combined, name="GLM_combined").to_string())


In [ ]:
# ============================================================
# POMDP (separate short+long), POMDP (combined), GLM (combined)
# -- same visual style as the cell-9 comparison above (grouped AIC/AICc/BIC
# bars, y_min-zoomed since raw AIC/AICc/BIC sit far from 0, LaTeX table
# export), just extended to a third bar group for the GLM. Shared with the
# commit-only comparison below via plot_pomdp_vs_glm_bic_comparison.
# ============================================================
from src.utils import plot_pomdp_vs_glm_bic_comparison

pomdp_separate_key = f"{best_model_short}+{best_model_long}"
pomdp_separate_row = results["metrics_table"].loc[pomdp_separate_key]
pomdp_combined_row = results["metrics_table"].loc[best_model_combined]

bic_table, fig, ax = plot_pomdp_vs_glm_bic_comparison(
    pomdp_combined_row,
    pomdp_separate_row,
    metrics_glm_combined,
    combined_name=best_model_combined,
    separate_name=pomdp_separate_key,
    export_table_path=os.path.join(report_tables_path, "pomdp_vs_glm_bic.tex"),
    fname="pomdp_combined_separate_vs_glm_combined",
    path=report_figures_path,
    is_latex=is_latex,
)

In [ ]:
# ============================================================
# POMDP (separate short+long), POMDP (combined), GLM (combined)
# -- "commit" framing of the comparison above. results_df_dict was loaded
# with commit=True (see the load_fitted_results call near the top), which
# points at data/POMDP_commit/ instead of data/POMDP/: those fits were
# optimized directly against log_likelihood_commit (decide vs. wait), not
# the full 3-way log_likelihood -- see the POMDP_COMMIT routing in
# src/config/schema.py and the cost-function selection in
# src/pomdp/pomdp.py's make_cost_function. So `results["metrics_table"]"`
# above is ALREADY the commit-only, GLM-comparable comparison; there is no
# separate rescoring to do here. (An earlier version of this cell reused
# BIC_commit/bic_commit.pkl -- built by notebooks/compute_bic_commit.py,
# which re-scores a *different*, full-LL-optimized fit under
# log_likelihood_commit without re-fitting. That pipeline is correct for
# the non-commit sibling notebook_comparison.ipynb, whose results_df_dict
# has no commit-fit params at all, but it doesn't apply here and gave a
# worse, mismatched-objective number.)
# ============================================================
pomdp_combined_row_commit = pomdp_combined_row
pomdp_separate_row_commit = pomdp_separate_row

bic_table_commit, fig, ax = plot_pomdp_vs_glm_bic_comparison(
    pomdp_combined_row_commit,
    pomdp_separate_row_commit,
    metrics_glm_combined,
    combined_name=best_model_combined,
    separate_name=pomdp_separate_key,
    export_table_path=os.path.join(report_tables_path, "pomdp_vs_glm_bic_commit.tex"),
    fname="pomdp_combined_separate_vs_glm_combined_commit",
    path=report_figures_path,
    is_latex=is_latex,
    title="Model selection metrics (commit: decide vs. wait)",
    legend_loc="upper left",
    legend_bbox_to_anchor=(0, 0),
)
plt.show()

## GLM vs POMDP (separate): `SBEXT-RP-----` paired with every long model

The comparison above only pairs the fixed best short model with the single
best long model (`best_model_long`, `LBEXT-RPHCLUK`). Here every available
long model is paired with the fixed best short model (`best_model_short`,
`SBEXT-RP-----`) instead, so all `SBEXT-RP-----` + long-model combinations
can be judged against the GLM at once, still on the commit (decide vs. wait)
likelihood so the POMDP and GLM are scored on the same target.

In [ ]:
# ============================================================
# GLM vs POMDP (separate short+long), sweeping ALL long models paired with
# the fixed best short model (SBEXT-RP-----). results_df_dict already holds
# the properly commit-fit after_lls_ga for every model (see the note in the
# cell above). Per-subject-summed convention throughout (each subject's own
# k/n_obs/ll, summed across subjects) -- matches metrics_glm_combined above
# and results["metrics_table"], rather than pooling ll/n_obs across subjects
# into one compute_metrics() call.
# ============================================================
from src.utils.plotting import (
    _set_plot_style, _save_figure, _texttt_with_condition_subscript,
    compute_metrics, _ensure_loglikelihoods, _per_subject_n_obs,
)

def per_subject_summed_metrics(dfs, n_params_list):
    """Sum compute_metrics(...) over subjects for one or more results_df's
    combined row-wise (e.g. a subject's short df row + their long df row),
    each contributing its own n_params to that subject's k."""
    totals = {"sum logL": 0.0, "AIC": 0.0, "AICc": 0.0, "BIC": 0.0}
    k_total = sum(n_params_list)
    n_subj = len(dfs[0])
    for i in range(n_subj):
        ll_i = 0.0
        n_obs_i = 0
        for df, n_params in zip(dfs, n_params_list):
            row = df.iloc[i]
            ll_raw = row["after_lls_ga"]
            sign = -1.0 if ll_raw > 0 else 1.0
            ll_i += sign * ll_raw
            n_obs_i += _per_subject_n_obs(row["data_dict_of_lists"])
        m_i = compute_metrics(ll_i, k_total, n_obs_i)
        for key in totals:
            totals[key] += m_i[key]
    totals["k"] = k_total
    totals["n_obs"] = sum(_per_subject_n_obs(row) for row in dfs[0]["data_dict_of_lists"])
    return totals

rows = {}
df_short_best = results_df_dict[best_model_short]
for task in long_models:
    df_task = results_df_dict[task]
    n_params_short = len(df_short_best["fit_params_ga"].iloc[0])
    n_params_task = len(df_task["fit_params_ga"].iloc[0])
    rows[f"{best_model_short}+{task}"] = per_subject_summed_metrics(
        [df_short_best, df_task], [n_params_short, n_params_task]
    )
rows["GLM"] = dict(metrics_glm_combined)

all_long_vs_glm_table = pd.DataFrame(rows).T.sort_values("BIC")
print(all_long_vs_glm_table.to_string())

export_path = os.path.join(report_tables_path, "pomdp_all_long_vs_glm_bic_commit.tex")
df_latex = all_long_vs_glm_table.reset_index().rename(columns={"index": "Model"})
df_latex["Model"] = df_latex["Model"].apply(
    lambda x: r"$\texttt{GLM}$" if x == "GLM" else f"${_texttt_with_condition_subscript(x)}$"
)
cols_order = ["Model", "sum logL", "k", "n_obs", "AIC", "AICc", "BIC"]
df_latex = df_latex[cols_order].rename(columns={"n_obs": r"$n_{\text{obs}}$", "k": r"$N_{\text{params}}$"})
df_latex.to_latex(export_path, index=False, float_format="%.2f", escape=False, column_format="lcccccc")
print(f"LaTeX table exported as {export_path}")

# Sorted horizontal bar chart, GLM highlighted, best (lowest BIC) at top.
_set_plot_style(font_size=12, is_latex=is_latex)
fig, ax = plt.subplots(figsize=(9, 7))
plot_df = all_long_vs_glm_table.sort_values("BIC")
colors = ["tab:green" if name == "GLM" else "tab:orange" for name in plot_df.index]
ax.barh(range(len(plot_df)), plot_df["BIC"], color=colors, alpha=0.9)
ax.set_yticks(range(len(plot_df)))
ax.set_yticklabels(plot_df.index, fontsize=9)
ax.invert_yaxis()
ax.set_xlabel("BIC (commit: decide vs. wait; lower = better)")
ax.set_title(f"GLM vs {best_model_short} + every long model")
# Log scale: a couple of long models fit orders of magnitude worse than the
# rest, and a linear axis would flatten every other bar to invisibility.
ax.set_xscale("log")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
_save_figure(fig, "pomdp_all_long_vs_glm_bic_commit", report_figures_path)
plt.show()

In [ ]:
# ============================================================
# Same sweep as above, sorted/exported/plotted by AIC instead of BIC.
# all_long_vs_glm_table already has AIC computed by compute_metrics, so this
# just re-sorts, re-exports, and re-plots on that criterion.
# ============================================================
all_long_vs_glm_table_aic = all_long_vs_glm_table.sort_values("AIC")
print(all_long_vs_glm_table_aic.to_string())

export_path_aic = os.path.join(report_tables_path, "pomdp_all_long_vs_glm_aic_commit.tex")
df_latex_aic = all_long_vs_glm_table_aic.reset_index().rename(columns={"index": "Model"})
df_latex_aic["Model"] = df_latex_aic["Model"].apply(
    lambda x: r"$\texttt{GLM}$" if x == "GLM" else f"${_texttt_with_condition_subscript(x)}$"
)
df_latex_aic = df_latex_aic[cols_order].rename(columns={"n_obs": r"$n_{\text{obs}}$", "k": r"$N_{\text{params}}$"})
df_latex_aic.to_latex(export_path_aic, index=False, float_format="%.2f", escape=False, column_format="lcccccc")
print(f"LaTeX table exported as {export_path_aic}")

# Sorted horizontal bar chart, GLM highlighted, best (lowest AIC) at top.
_set_plot_style(font_size=12, is_latex=is_latex)
fig, ax = plt.subplots(figsize=(9, 7))
colors_aic = ["tab:green" if name == "GLM" else "tab:orange" for name in all_long_vs_glm_table_aic.index]
ax.barh(range(len(all_long_vs_glm_table_aic)), all_long_vs_glm_table_aic["AIC"], color=colors_aic, alpha=0.9)
ax.set_yticks(range(len(all_long_vs_glm_table_aic)))
ax.set_yticklabels(all_long_vs_glm_table_aic.index, fontsize=9)
ax.invert_yaxis()
ax.set_xlabel("AIC (commit: decide vs. wait; lower = better)")
ax.set_title(f"GLM vs {best_model_short} + every long model")
# Log scale: a couple of long models fit orders of magnitude worse than the
# rest, and a linear axis would flatten every other bar to invisibility.
ax.set_xscale("log")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
_save_figure(fig, "pomdp_all_long_vs_glm_aic_commit", report_figures_path)
plt.show()

# Bic score per subject and look at the indvidiual trials as well. 

# BIC Score for the GLM that is from the POMDP data not on human 

## Per-subject BIC: GLM (combined) vs POMDP (combined)

The section above compares *aggregate* BIC (summed log-likelihood and parameter count across all subjects, one shared `n_obs`). This section computes one BIC per subject for each model (`BIC_i = k*log(n_obs_i) - 2*logL_i`, using that subject's own `n_obs`), so the comparison can be a paired test across subjects (does GLM or POMDP win for *this* subject) rather than one pooled number. Same caveat as above: POMDP's `n_obs` counts trials/games from `data_dict_of_lists`, the GLM's counts decide/wait draws -- both log-likelihoods are summed at the per-draw level, so this is the closest available per-subject comparison, not a perfectly identical unit of observation.

In [ ]:
# Per-subject POMDP (combined model) BIC. results_df_dict was loaded with
# commit=True, so after_lls_ga IS ALREADY the commit (decide vs wait)
# log-likelihood these params were fit to maximize (see the note in the
# aggregate-BIC section above) -- no bic_commit.pkl merge needed.
from scipy.stats import ttest_rel, wilcoxon

df_pomdp_combined_raw = results_df_dict[best_model_combined]

ll_raw = df_pomdp_combined_raw["after_lls_ga"].astype(float).values
commit_sign = -1.0 if np.nanmean(ll_raw) > 0 else 1.0
ll_commit_signed = commit_sign * ll_raw

k_pomdp = len(df_pomdp_combined_raw["fit_params_ga"].iloc[0])
n_obs_pomdp = df_pomdp_combined_raw["data_dict_of_lists"].apply(
    lambda d: sum(
        len(seq)
        for horizon_df in d.values()
        for seq in horizon_df["draw_yellow_blue_action_outcome"].values
    )
).values

bic_pomdp_per_subject = k_pomdp * np.log(n_obs_pomdp) - 2 * ll_commit_signed

df_pomdp_bic = pd.DataFrame({
    "subject_ID": df_pomdp_combined_raw["subject_ID"].values,
    "BIC_pomdp": bic_pomdp_per_subject,
    "n_obs_pomdp": n_obs_pomdp,
}).dropna(subset=["BIC_pomdp"])

# Per-subject GLM (combined) BIC -- ll_raw and n_obs_raw from cell 16
# cover all raw draws, not just the valid_mask-filtered subset.
N_PARAMS_GLM = 7
df_glm_bic = pd.DataFrame([
    {
        "subject_ID": uid,
        "BIC_glm": N_PARAMS_GLM * np.log(b["n_obs_raw"]) - 2 * b["ll_raw"],
        "n_obs_glm": b["n_obs_raw"],
    }
    for uid, b in glm_combined_by_id.items()
])

df_bic_per_subject = df_pomdp_bic.merge(df_glm_bic, on="subject_ID", how="inner")
diff = df_bic_per_subject["BIC_pomdp"] - df_bic_per_subject["BIC_glm"]

print(f"N subjects compared: {len(df_bic_per_subject)}")
print(f"POMDP BIC (commit): {df_bic_per_subject['BIC_pomdp'].mean():.2f} ± {df_bic_per_subject['BIC_pomdp'].std():.2f}")
print(f"GLM   BIC:          {df_bic_per_subject['BIC_glm'].mean():.2f} ± {df_bic_per_subject['BIC_glm'].std():.2f}")
print(f"Mean paired difference (POMDP - GLM): {diff.mean():+.2f}")

t_stat, t_p = ttest_rel(df_bic_per_subject["BIC_pomdp"], df_bic_per_subject["BIC_glm"])
w_stat, w_p = wilcoxon(df_bic_per_subject["BIC_pomdp"], df_bic_per_subject["BIC_glm"])
print(f"Paired t-test:        t={t_stat:.3f}, p={t_p:.4f}")
print(f"Wilcoxon signed-rank: W={w_stat:.1f}, p={w_p:.4f}")
print(f"GLM lower-BIC (better) on {int((diff > 0).sum())}/{len(diff)} subjects; "
      f"POMDP lower-BIC (better) on {int((diff < 0).sum())}/{len(diff)} subjects")


In [ ]:
from src.utils.plotting import _set_plot_style

_set_plot_style(font_size=12, is_latex=is_latex)
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ax = axes[0]
ax.scatter(
    df_bic_per_subject["BIC_pomdp"], df_bic_per_subject["BIC_glm"],
    alpha=0.6, s=60, color="steelblue", edgecolors="black",
)
lim = [
    min(df_bic_per_subject["BIC_pomdp"].min(), df_bic_per_subject["BIC_glm"].min()),
    max(df_bic_per_subject["BIC_pomdp"].max(), df_bic_per_subject["BIC_glm"].max()),
]
ax.plot(lim, lim, "k--", alpha=0.5, label="y = x")
ax.set_xlabel("POMDP BIC (per subject)")
ax.set_ylabel("GLM BIC (per subject)")
ax.set_title("Per-subject BIC: POMDP vs GLM (lower = better)")
ax.legend()
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax2 = axes[1]
ax2.boxplot(
    [df_bic_per_subject["BIC_glm"], df_bic_per_subject["BIC_pomdp"]],
    tick_labels=["GLM", "POMDP"],
)
ax2.set_ylabel("BIC (per subject, lower = better)")
ax2.set_title(f"Paired comparison (p={w_p:.4f}, Wilcoxon)")
ax2.spines["top"].set_visible(False)
ax2.spines["right"].set_visible(False)

fig.tight_layout()
fig.savefig(os.path.join(report_figures_path, "pomdp_vs_glm_bic_per_subject.pdf"), bbox_inches="tight")
plt.show()

## Personalized (per-subject) model selection vs. a single fixed model

Every section above picks *one* model structure per horizon and applies it to
all 105 subjects (the usual approach: aggregate BIC, summed log-likelihood
and one shared `n_obs`, picks a single winner). `notebooks/per_subject_model_selection_commit.py`
asks a different question: if every subject were free to be explained by
*whichever* commit-fit model structure suits them individually (own BIC,
own `n_obs`), how much of the population is actually better described by a
different mechanism set than the population-level "best" model -- and does a
personalized ensemble outperform the single fixed model in cumulative BIC?

For the combined ("C") horizon, GLM is included as one more per-subject
candidate alongside every POMDP(commit) config (GLM has no separate-horizon
version, so it only enters here -- see the note above `per_subject_model_selection_commit.py`).

**Aggregation convention** (see that script's docstring for the full
rationale): cumulative BIC here is `Sum_i(k_i*log(n_obs_i) - 2*ll_i)`, i.e.
each subject's *own* BIC (own model, own `n_obs`) summed across subjects --
the exact BIC of "the ensemble of 105 independently-selected models." The
"fixed model" comparison number is computed the *same* way (that subject's
`n_obs`, but scored under the single population-level winner), so the two
numbers differ only in whether model structure is personalized, not in the
aggregation formula. This is **not** the same convention as the aggregate
BIC tables above (which use one shared `n_obs` in a single `log()` term) --
don't compare these numbers directly against `results_short`/`results_long`/`results_combined`.

In [ ]:
from src.utils.plotting import _texttt_with_condition_subscript

personalized_dir = os.path.join(project_root, "data", "POMDP_commit")
horizon_labels = {"S": "Short", "L": "Long", "C": "Combined"}

personalized_summary_rows = []
per_subject_selection_dfs = {}
for h in ["S", "L", "C"]:
    df_h = pd.read_csv(os.path.join(personalized_dir, f"per_subject_model_selection_commit_{h}.csv"))
    per_subject_selection_dfs[h] = df_h
    both = df_h.dropna(subset=["fixed_model_BIC"])
    row = {
        "Horizon": horizon_labels[h],
        "N subjects": len(both),
        "Fixed model": both["fixed_model_task"].iloc[0],
        "Personalized cumulative BIC": both["winning_BIC"].sum(),
        "Fixed cumulative BIC": both["fixed_model_BIC"].sum(),
    }
    if "GLM_BIC" in df_h.columns:
        both_glm = df_h.dropna(subset=["GLM_BIC"])
        row["GLM cumulative BIC"] = both_glm["GLM_BIC"].sum()
    row["Delta (fixed - personalized)"] = row["Fixed cumulative BIC"] - row["Personalized cumulative BIC"]
    row["N subjects improved (>2 BIC)"] = int((both["fixed_model_BIC"] - both["winning_BIC"] > 2).sum())
    personalized_summary_rows.append(row)

personalized_summary = pd.DataFrame(personalized_summary_rows).set_index("Horizon")
print(personalized_summary.to_string())

# LaTeX export (booktabs style, matching the rest of this notebook's tables).
# na_rep="--" -- GLM cumulative BIC is only defined for Combined (GLM has no
# separate-horizon version), so Short/Long leave that cell blank rather than
# printing a literal "NaN" into the paper table.
df_latex_personalized = personalized_summary.reset_index()
df_latex_personalized["Fixed model"] = df_latex_personalized["Fixed model"].apply(
    lambda x: f"${_texttt_with_condition_subscript(x)}$"
)
export_path_personalized = os.path.join(report_tables_path, "personalized_model_selection_commit.tex")
df_latex_personalized.to_latex(
    export_path_personalized, index=False, float_format="%.2f", escape=False, na_rep="--",
    column_format="l" + "c" * (len(df_latex_personalized.columns) - 1),
)
print(f"\nLaTeX table exported as {export_path_personalized}")


In [ ]:
# Grouped bar chart: Personalized vs Fixed (vs GLM alone for Combined),
# one group per horizon. Values span a wide range across horizons, so each
# horizon gets its own y-axis via subplots rather than one shared axis.
_set_plot_style(font_size=12, is_latex=is_latex)
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, h in zip(axes, ["S", "L", "C"]):
    row = personalized_summary.loc[horizon_labels[h]]
    labels = ["Personalized\nselection", f"Fixed\n({row['Fixed model']})"]
    values = [row["Personalized cumulative BIC"], row["Fixed cumulative BIC"]]
    colors = ["tab:blue", "tab:orange"]
    ax.bar(labels, values, color=colors, alpha=0.9)
    ax.set_title(f"{horizon_labels[h]} (n={int(row['N subjects'])})")
    ax.set_ylabel("Cumulative BIC (per-subject-summed)")
    ax.tick_params(axis="x", labelsize=9)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ymin = min(values) * 0.98
    ax.set_ylim(bottom=ymin)

fig.suptitle("Personalized per-subject model selection vs. a single fixed model (commit likelihood)")
fig.tight_layout()
_save_figure(fig, "personalized_model_selection_commit", report_figures_path)
plt.show()


In [ ]:
# Per-subject scatter: personalized-winner BIC vs. fixed-model BIC, one
# panel per horizon. Points below the y=x line are subjects better described
# by their own selected model than by the population-level winner.
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, h in zip(axes, ["S", "L", "C"]):
    df_h = per_subject_selection_dfs[h].dropna(subset=["fixed_model_BIC"])
    ax.scatter(df_h["fixed_model_BIC"], df_h["winning_BIC"], alpha=0.6, s=40,
               color="steelblue", edgecolors="black")
    lim = [
        min(df_h["fixed_model_BIC"].min(), df_h["winning_BIC"].min()),
        max(df_h["fixed_model_BIC"].max(), df_h["winning_BIC"].max()),
    ]
    ax.plot(lim, lim, "k--", alpha=0.5, label="y = x")
    ax.set_xlabel("Fixed model BIC (per subject)")
    ax.set_ylabel("Personalized-winner BIC (per subject)")
    ax.set_title(horizon_labels[h])
    ax.legend(fontsize=9)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

fig.suptitle("Per-subject BIC: personalized winner vs. fixed aggregate-best model (commit likelihood)")
fig.tight_layout()
_save_figure(fig, "personalized_vs_fixed_bic_per_subject_commit", report_figures_path)
plt.show()
